# Burn cost demo: inspect the SQL results

Read the real SQL Server rows written by the other notebooks. You can run this after 03, after promotion in 06, and after the weekly run in 07.
This notebook only reads the database. The final cell exports the tables and views to Excel.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sqlalchemy import text

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "pricing_models").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from demo_sql_runtime import get_engine

engine = get_engine()
with engine.connect() as connection:
    print(connection.execute(text("SELECT @@VERSION")).scalar_one())
    print("Database:", connection.execute(text("SELECT DB_NAME()")).scalar_one())


def read(query, **params):
    with engine.connect() as connection:
        return pd.read_sql_query(text(query), connection, params=params)

## Champion and challengers

After 03 there is one candidate. After 06 it is the champion. After 07 there are also three challengers.
Their definition revision stays the same because these refits use the same declared model recipe.


In [ ]:
registry = read(
    """
    SELECT model_name, role, definition_revision, refit_type, data_as_of_date,
           package_version, model_run_id, rate_package_id, deployment_slot
    FROM pricing.V_MODEL_REGISTRY
    WHERE model_name = :model AND (deployment_slot = :slot OR deployment_slot IS NULL)
    ORDER BY package_version
""",
    model="DEMO_BURN_COST",
    slot="DEMO_BURN_COST_ONLY",
)
display(registry)

## The compressed recipe and its readable JSON

`recipe_gzip` is the stored binary value. The hex preview below comes from SQL Server.
`recipe_json` is calculated when selected; SQL does not store a second plain-text copy.
The byte counts below are also measured by SQL Server.


In [ ]:
recipes = read(
    """
    SELECT recipe.recipe_id, recipe.recipe_revision,
           DATALENGTH(recipe.recipe_json) AS original_text_bytes,
           DATALENGTH(recipe.recipe_gzip) AS stored_gzip_bytes,
           recipe.recipe_gzip, recipe.recipe_json
    FROM pricing.MODEL_RECIPE AS recipe
    JOIN pricing.PRICING_MODEL AS model ON model.model_id = recipe.model_id
    WHERE model.model_name = :model
    ORDER BY recipe.recipe_revision
""",
    model="DEMO_BURN_COST",
)
recipe_display = recipes.drop(columns=["recipe_gzip"]).copy()
recipe_display["recipe_gzip_hex_preview"] = recipes.recipe_gzip.map(
    lambda value: "0x" + bytes(value).hex()[:96].upper() + "..."
)
recipe_display["storage_saved_percent"] = (
    100 * (1 - recipes.stored_gzip_bytes / recipes.original_text_bytes)
).round(1)
display(recipe_display)

columns = read("""
    SELECT name, is_computed, is_persisted, definition
    FROM sys.computed_columns
    WHERE object_id = OBJECT_ID('pricing.MODEL_RECIPE')
""")
display(columns)

## Read one decoded recipe

These are the groups, specials and settings from 02, now read back from SQL.

In [ ]:
import json

if recipes.empty:
    print("Run 03 to save a recipe first.")
else:
    print(json.dumps(json.loads(recipes.iloc[-1].recipe_json), indent=2, ensure_ascii=False))

## Monitoring evidence

Notebook 07 writes four observations and three challenger packages. The champion score does not create another package.

In [ ]:
observations = read(
    """
    SELECT run.*
    FROM mlops.MODEL_MONITOR_RUN AS run
    JOIN pricing.PRICING_MODEL AS model ON model.model_id = run.model_id
    WHERE model.model_name = :model
    ORDER BY run.started_ts, run.variant_code
""",
    model="DEMO_BURN_COST",
)
display(observations)

metrics = read(
    """
    SELECT run.variant_code, metric.metric_name, metric.metric_value
    FROM mlops.MODEL_MONITOR_METRIC AS metric
    JOIN mlops.MODEL_MONITOR_RUN AS run ON run.monitor_run_id = metric.monitor_run_id
    JOIN pricing.PRICING_MODEL AS model ON model.model_id = run.model_id
    WHERE model.model_name = :model
    ORDER BY run.variant_code, metric.metric_name
""",
    model="DEMO_BURN_COST",
)
display(metrics)

## Relativities

The two views give the detailed model configuration and the fitted relativities. This preview limits each to 20 rows; the workbook includes other SQL tables and views too.

In [ ]:
for view in ["V_FINAL_MODEL_RELATIVITY", "V_MODEL_RELATIVITY"]:
    print("pricing." + view)
    display(
        read(
            f"SELECT TOP (20) * FROM pricing.{view} WHERE model_name = :model",
            model="DEMO_BURN_COST",
        )
    )

## Export the actual tables and views

This exports at most 20 rows per object from this dedicated demo database. The index includes full row counts.
Long JSON values have separate files beside the workbook. Binary recipe values appear as hex, like SSMS.
Each export gets its own timestamped directory.


In [ ]:
from export_sql import export_sql_tables

workbook = export_sql_tables(engine)
print(workbook)